## Converting the data into chatML like format in jsonl file

- This is doing for fine tuning llm on the data

In [ ]:
import json
import random
from datasets import load_dataset
from rank_bm25 import BM25Okapi

In [48]:
data = load_dataset("umarbutler/better-cuad", split="train").to_pandas()

In [49]:
data.columns

Index(['Filename', 'Document Name', 'Document Name-Answer', 'Parties',
       'Parties-Answer', 'Agreement Date', 'Agreement Date-Answer',
       'Effective Date', 'Effective Date-Answer', 'Expiration Date',
       'Expiration Date-Answer', 'Renewal Term', 'Renewal Term-Answer',
       'Notice Period To Terminate Renewal',
       'Notice Period To Terminate Renewal- Answer', 'Governing Law',
       'Governing Law-Answer', 'Most Favored Nation',
       'Most Favored Nation-Answer', 'Competitive Restriction Exception',
       'Competitive Restriction Exception-Answer', 'Non-Compete',
       'Non-Compete-Answer', 'Exclusivity', 'Exclusivity-Answer',
       'No-Solicit Of Customers', 'No-Solicit Of Customers-Answer',
       'No-Solicit Of Employees', 'No-Solicit Of Employees-Answer',
       'Non-Disparagement', 'Non-Disparagement-Answer',
       'Termination For Convenience', 'Termination For Convenience-Answer',
       'Rofr/Rofo/Rofn', 'Rofr/Rofo/Rofn-Answer', 'Change Of Control',
      

- Extracting only useful columns

In [50]:
columns_to_keep = [
    'Document Name-Answer', 
    'Parties-Answer', 
    'Agreement Date-Answer', 
    'Effective Date-Answer', 
    'Expiration Date-Answer', 
    'Governing Law-Answer', 
    'Text'
]

data = data[columns_to_keep]

- Removing Answer from last of columns so it won't be a confusion in chat

In [51]:
data.columns = data.columns.str.strip().str.removesuffix('-Answer')

print(data.columns)

Index(['Document Name', 'Parties', 'Agreement Date', 'Effective Date',
       'Expiration Date', 'Governing Law', 'Text'],
      dtype='str')


In [52]:
data.head(3)

,Document Name,Parties,Agreement Date,Effective Date,Expiration Date,Governing Law,Text
0,MARKETING AFFILIATE AGREEMENT,"Birch First Global Investments Inc. (""Company""...",5/8/14,NaN,12/31/14,Nevada,Exhibit 10.27\n\nMARKETING AFFILIATE AGREEMENT...
1,VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"Rogers Cable Communications Inc. (""Rogers""); E...",7/11/06,7/11/06,6/30/10,"Ontario, Canada",Exhibit 10.B.01 EXECUTION COPY\n\nVIDEO-ON-DEM...
2,CONTENT DISTRIBUTION AND LICENSE AGREEMENT,"CONVERGTV, INC. (“ConvergTV”); Fulucai Product...",11/15/12,11/15/12,NaN,Florida,CONTENT DISTRIBUTION AND LICENSE AGREEMENT D...


In [53]:
data.isnull().sum()

Document Name        0
Parties              1
Agreement Date      45
Effective Date     151
Expiration Date    181
Governing Law       76
Text                 0
dtype: int64

- Converting null values to 'Not found' so model understands if there is nothing in text it can be null or not found instead of creating synthetic values (basically hallucinating)

In [54]:
fill_dict = {col: 'Not found' for col in data.columns}

data = data.fillna(value=fill_dict)

In [55]:
data.isnull().sum()

Document Name      0
Parties            0
Agreement Date     0
Effective Date     0
Expiration Date    0
Governing Law      0
Text               0
dtype: int64

- Final conversion to jsonl
- We are extracting chunks from the text, matching using BM25 algorithm (using best 3 matches) so we don't have to train model on all data which would make training absolute slow
- Also adding first 3 chunks without match because of chance of leaving exact important data chances & this strategy would prevent it from happening

In [56]:
target_columns = [col for col in data.columns if col != 'Text']

jsonl_output = []

for index, row in data.iterrows():
    raw_contract_text = str(row['Text']).strip()
    raw_contract_text = raw_contract_text.replace("\u2028", " ").replace("\u2029", " ")

    chunks = [c.strip() for c in raw_contract_text.split('\n\n') if len(c.strip()) > 20]
    
    if len(chunks) < 5:
        chunks = [c.strip() for c in raw_contract_text.split('\n') if len(c.strip()) > 20]

    preamble_chunks = chunks[:3]
    
    remaining_chunks = chunks[3:]

    use_bm25 = len(remaining_chunks) > 0

    if use_bm25:
        tokenized_chunks = [chunk.lower().split(" ") for chunk in remaining_chunks]
        bm25 = BM25Okapi(tokenized_chunks)
    
    for clause_name in target_columns:
        target_answer = str(row[clause_name]).strip()
        
        if use_bm25:
            tokenized_query = clause_name.lower().split(" ")
            top_n_chunks = bm25.get_top_n(tokenized_query, remaining_chunks, n=3)
            final_context_list = preamble_chunks + top_n_chunks
        else:
            final_context_list = preamble_chunks

        retrieved_context = "\n\n".join(final_context_list)
        
        prompt_dict = {
            "messages": [
                {
                    "role": "system", 
                    "content": "You are a legal data extraction AI. Extract the requested entity from the text. If it is not present, output strictly 'Not found'."
                },
                {
                    "role": "user", 
                    "content": f"Extract the exact value for '{clause_name}'.\n\nContract Text:\n{retrieved_context}"
                },
                {
                    "role": "assistant", 
                    "content": target_answer
                }
            ]
        }
        
        jsonl_output.append(prompt_dict)

random.seed(42) 
random.shuffle(jsonl_output)

output_filename = "phi4_legal_extraction_data.jsonl"
with open(output_filename, "w", encoding="utf-8") as f:
    for item in jsonl_output:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Successfully generated {len(jsonl_output)} dynamic training examples.")

Successfully generated 3060 dynamic training examples.


- Analysing the tokens in each sample of jsonl so we know the exact limit

In [57]:
import tiktoken

# Phi-4-mini models use the o200k_base tiktoken tokenizer
encoding = tiktoken.get_encoding("o200k_base")

token_list = []

with open("phi4_legal_extraction_data.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if line.strip():
            tokens = len(encoding.encode(line))
            token_list.append(tokens)
            print(f"Sample {i}: {tokens} tokens")


Sample 1: 386 tokens
Sample 2: 563 tokens
Sample 3: 378 tokens
Sample 4: 311 tokens
Sample 5: 445 tokens
Sample 6: 249 tokens
Sample 7: 2397 tokens
Sample 8: 422 tokens
Sample 9: 544 tokens
Sample 10: 1564 tokens
Sample 11: 1762 tokens
Sample 12: 3134 tokens
Sample 13: 571 tokens
Sample 14: 2197 tokens
Sample 15: 3056 tokens
Sample 16: 234 tokens
Sample 17: 310 tokens
Sample 18: 189 tokens
Sample 19: 2501 tokens
Sample 20: 317 tokens
Sample 21: 1403 tokens
Sample 22: 3509 tokens
Sample 23: 221 tokens
Sample 24: 871 tokens
Sample 25: 2005 tokens
Sample 26: 242 tokens
Sample 27: 2290 tokens
Sample 28: 384 tokens
Sample 29: 441 tokens
Sample 30: 519 tokens
Sample 31: 806 tokens
Sample 32: 401 tokens
Sample 33: 467 tokens
Sample 34: 249 tokens
Sample 35: 388 tokens
Sample 36: 562 tokens
Sample 37: 1412 tokens
Sample 38: 368 tokens
Sample 39: 1580 tokens
Sample 40: 378 tokens
Sample 41: 752 tokens
Sample 42: 305 tokens
Sample 43: 277 tokens
Sample 44: 2564 tokens
Sample 45: 1256 tokens
Samp

In [58]:
import numpy as np

token_count_arr = np.array(token_list)

In [59]:
print(f"""
Maximum token count in a sample - {token_count_arr.max()}
Minimum token count in a sample - {token_count_arr.min()}
Average token count in all samples - {token_count_arr.mean()}
Total token count in all samples - {token_count_arr.sum()}
""")


Maximum token count in a sample - 8583
Minimum token count in a sample - 153
Average token count in all samples - 1137.088888888889
Total token count in all samples - 3479492



In [67]:
print(f'''
Tokens Greater than 8000: {np.count_nonzero(token_count_arr > 8000)}
Tokens 7001 to 8000:     {np.count_nonzero((token_count_arr > 7000) & (token_count_arr <= 8000))}
Tokens 6001 to 7000:     {np.count_nonzero((token_count_arr > 6000) & (token_count_arr <= 7000))}
Tokens 5001 to 6000:     {np.count_nonzero((token_count_arr > 5000) & (token_count_arr <= 6000))}
Tokens 4001 to 5000:     {np.count_nonzero((token_count_arr > 4001) & (token_count_arr <= 5000))}
Tokens 3001 to 4000:     {np.count_nonzero((token_count_arr > 3000) & (token_count_arr <= 4000))}
Tokens 2001 to 3000:     {np.count_nonzero((token_count_arr > 2000) & (token_count_arr <= 3000))}
Tokens 1001 to 2000:     {np.count_nonzero((token_count_arr > 1000) & (token_count_arr <= 2000))}
Tokens 501 to 1000:      {np.count_nonzero((token_count_arr > 500) & (token_count_arr <= 1000))}
Tokens 251 to 500:       {np.count_nonzero((token_count_arr > 250) & (token_count_arr <= 500))}
Tokens 0 to 250:         {np.count_nonzero(token_count_arr <=250)}
''')



Tokens Greater than 8000: 2
Tokens 7001 to 8000:     4
Tokens 6001 to 7000:     2
Tokens 5001 to 6000:     14
Tokens 4001 to 5000:     46
Tokens 3001 to 4000:     197
Tokens 2001 to 3000:     391
Tokens 1001 to 2000:     395
Tokens 501 to 1000:      639
Tokens 251 to 500:       1187
Tokens 0 to 250:         183



- Now data is ready